<a href="https://colab.research.google.com/github/stangler/jp-natural-voice-app/blob/main/StyleBertVITS2_hirakawa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# セル0: GPU確認
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024 // 1024} MiB')

Fri Apr  3 05:45:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# セル1: Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# セル2: リポジトリクローン
import os
if not os.path.exists('/content/jp-natural-voice-app'):
    !git clone https://github.com/stangler/jp-natural-voice-app /content/jp-natural-voice-app
else:
    print('Already cloned')
os.chdir('/content/jp-natural-voice-app')

Cloning into '/content/jp-natural-voice-app'...
remote: Enumerating objects: 6630, done.
remote: Counting objects: 100% (6630/6630), done.
remote: Compressing objects: 100% (2191/2191), done.
remote: Total 6630 (delta 4380), reused 6591 (delta 4353), pack-reused 0 (from 0)
Receiving objects: 100% (6630/6630), 14.64 MiB | 17.04 MiB/s, done.
Resolving deltas: 100% (4380/4380), done.


In [5]:
# セル3: パッケージセットアップ（固定バージョン）
!pip install torch==2.4.0+cu124 torchaudio==2.4.0+cu124 --index-url https://download.pytorch.org/whl/cu124 -q
!pip install transformers==4.37.2 huggingface_hub==0.19.4 numpy==1.26.4 -q
!pip install -r requirements.txt -q
print('パッケージセットアップ完了 ✅')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 50.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 87.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 61.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 MB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.4/128.4 MB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
# セル3.5: 【永続パッチ】jax/transformers 競合回避（セル3実行後は毎回実行）
filepath = '/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py'
with open(filepath, 'r') as f:
    lines = f.readlines()
new_lines = []
for line in lines:
    if 'import jax.numpy as jnp' in line:
        line = line.replace('import jax.numpy as jnp', 'pass  # jax disabled')
    new_lines.append(line)
with open(filepath, 'w') as f:
    f.writelines(new_lines)
print('jaxパッチ適用済み ✅')

jaxパッチ適用済み ✅


In [7]:
# セル4: bert_feature.py パッチ
# word2ph と text の長さ不一致による assert エラーを trim/pad で回避
patch_code = """
path = 'style_bert_vits2/nlp/japanese/bert_feature.py'
with open(path, 'r') as f:
    content = f.read()

old = '    text = text.replace(\"。\", \".\").replace(\"、\", \",\")\\n    assert len(word2ph) == len(text) + 2, text'

new = '''    text = text.replace(\"。\", \".\").replace(\"、\", \",\")
    expected = len(text) + 2
    if len(word2ph) != expected:
        if len(word2ph) > expected:
            word2ph = word2ph[:expected]
        else:
            word2ph = word2ph + [1] * (expected - len(word2ph))'''

count = content.count(old)
print(f'Found {count} occurrence(s)')
if count > 0:
    content = content.replace(old, new)
    with open(path, 'w') as f:
        f.write(content)
    print('bert_feature.py: Patched!')
else:
    print('bert_feature.py: Pattern not found (already patched or changed)')
"""

import subprocess
result = subprocess.run(['python3', '-c', patch_code], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

Found 0 occurrence(s)
bert_feature.py: Pattern not found (already patched or changed)



In [8]:
# セル5: lightning_fabric パッチ（毎回適用）
import importlib.util
if importlib.util.find_spec('lightning_fabric') is not None:
    import lightning_fabric
    lf_path = lightning_fabric.__file__.replace('__init__.py', 'utilities/cloud_io.py')
    with open(lf_path, 'r') as f:
        content = f.read()
    old = 'weights_only=True'
    new = 'weights_only=False'
    if old in content:
        content = content.replace(old, new)
        with open(lf_path, 'w') as f:
            f.write(content)
        print('lightning_fabric: Patched!')
    else:
        print('lightning_fabric: Already patched or not needed')
else:
    print('lightning_fabric not found - skip')

lightning_fabric not found - skip


In [9]:
# セル6: Drive からデータコピー
import os

# Drive の音声データ確認
DRIVE_WAV_PATH = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample'
wav_count = !find {DRIVE_WAV_PATH}/wavs -name '*.wav' 2>/dev/null | wc -l
esd_exists = os.path.exists(f'{DRIVE_WAV_PATH}/esd.list')
print(f'音声ファイル数: {wav_count[0]}')
print(f'esd.list: {esd_exists}')

# Data ディレクトリにコピー
os.makedirs('Data/hirakawa_sample/wavs', exist_ok=True)
!cp -n {DRIVE_WAV_PATH}/wavs/*.wav Data/hirakawa_sample/wavs/ 2>/dev/null || true
!cp -n {DRIVE_WAV_PATH}/esd.list Data/hirakawa_sample/esd.list 2>/dev/null || true

copied = !find Data/hirakawa_sample/wavs -name '*.wav' | wc -l
print(f'コピー済み wav: {copied[0]} ファイル ✅')

# Drive に前処理済みファイルがある場合はコピー（スキップ可）
DRIVE_BERT_PATH = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample_preprocessed'
if os.path.exists(DRIVE_BERT_PATH):
    print('前処理済みファイルが Drive に見つかりました。コピーします...')
    !cp {DRIVE_BERT_PATH}/*.bert.pt Data/hirakawa_sample/wavs/ 2>/dev/null || true
    !cp {DRIVE_BERT_PATH}/*.npy Data/hirakawa_sample/wavs/ 2>/dev/null || true
    bert_count = !find Data/hirakawa_sample/ -name '*.bert.pt' | wc -l
    npy_count = !find Data/hirakawa_sample/ -name '*.npy' | wc -l
    print(f'.bert.pt: {bert_count[0]}, .npy: {npy_count[0]}')
else:
    print('前処理済みファイルなし → 次のセルで前処理を実行してください')

音声ファイル数: 188
esd.list: True
コピー済み wav: 188 ファイル ✅
前処理済みファイルなし → 次のセルで前処理を実行してください


In [14]:
pip install huggingface_hub==0.24.0


In [15]:
pip install loguru


  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.8 MB/s eta 0:00:00


In [17]:
pip install onnxruntime


  Using cached onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 70.0 MB/s eta 0:00:00


In [22]:
!sed -n '1,200p /content/jp-natural-voice-app/preprocess_all.py


/bin/bash: -c: line 1: unexpected EOF while looking for matching `''
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [24]:
!grep pyopenjtalk_worker.initialize_worker /content/jp-natural-voice-app/preprocess_all.py


# pyopenjtalk_worker.initialize_worker()


In [26]:
pip install pyopenjtalk


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyopenjtalk: filename=pyopenjtalk-0.4.1-cp312-cp312-linux_x86_64.whl size=5764697 sha256=869ff8be138f90036ec8db9db725826fa5c784df4f1507c17972b7f51c03b928
  Stored in directory: /root/.cache/pip/wheels/7e/63/4a/2a68afc3843e86a32e5772b75f2652efe73aa9f4a8435a47d3
Successfully built pyopenjtalk


In [41]:
!pip install -q huggingface_hub


In [43]:
!huggingface-cli login



    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: read).
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in c

In [44]:
!cd /content/jp-natural-voice-app/pretrained && git clone https://huggingface.co/cl-tohoku/bert-base-japanese-v3 jp
!cd /content/jp-natural-voice-app/pretrained && git clone https://huggingface.co/bert-base-uncased en
!cd /content/jp-natural-voice-app/pretrained && git clone https://huggingface.co/bert-base-chinese zh


Cloning into 'jp'...
remote: Enumerating objects: 15, done.
remote: Total 15 (delta 0), reused 0 (delta 0), pack-reused 15 (from 1)
Receiving objects: 100% (15/15), 119.01 KiB | 1.59 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Filtering content: 100% (3/3), 1.34 GiB | 56.79 MiB/s, done.
Cloning into 'en'...
remote: Enumerating objects: 85, done.
remote: Total 85 (delta 0), reused 0 (delta 0), pack-reused 85 (from 1)
Receiving objects: 100% (85/85), 330.60 KiB | 10.33 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Filtering content: 100% (7/7), 3.21 GiB | 52.64 MiB/s, done.
Cloning into 'zh'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 58 (delta 1), reused 0 (delta 0), pack-reused 52 (from 1)
Receiving objects: 100% (58/58), 160.48 KiB | 3.34 MiB/s, done.
Resolving deltas: 100% (20/20), done.
Filtering content: 100% (4/4), 1.59 GiB | 37.56 MiB/s, done.


In [45]:
!ls /content/jp-natural-voice-app/pretrained


en  jp	zh


In [47]:
!pip install pyloudnorm


  Using cached pyloudnorm-0.2.0-py3-none-any.whl.metadata (6.6 kB)


In [51]:
!mkdir -p Data/hirakawa_sample/raw
!mv Data/hirakawa_sample/wavs/*.wav Data/hirakawa_sample/raw/


In [57]:
!mkdir -p /content/jp-natural-voice-app/Data/hirakawa_sample/text


In [58]:
!for f in /content/jp-natural-voice-app/Data/hirakawa_sample/raw/*.wav; do \
  name=$(basename "$f" .wav); \
  echo "これは音声合成モデルの学習用ダミーテキストです。" > /content/jp-natural-voice-app/Data/hirakawa_sample/text/${name}.txt; \
done


In [59]:
!ls /content/jp-natural-voice-app/Data/hirakawa_sample/text | head


hirakawa-0.txt
hirakawa-100.txt
hirakawa-101.txt
hirakawa-102.txt
hirakawa-103.txt
hirakawa-104.txt
hirakawa-105.txt
hirakawa-106.txt
hirakawa-107.txt
hirakawa-108.txt


In [48]:
# セル7: config.json 生成
import os
os.chdir('/content/jp-natural-voice-app')

!python preprocess_all.py \
    --model_name hirakawa_sample \
    --batch_size 4 \
    --epochs 100

print('config.json 生成確認:', os.path.exists('Data/hirakawa_sample/config.json'))

reading /content/jp-natural-voice-app/dict_data/user.dict_csv-4986e76d-9154-438a-a623-a2795f88da3f.tmp ... 5
emitting double-array: 100% |###########################################| 

done!
04-03 07:20:00 |  INFO  | train.py:72 | Step 1: start initialization...
model_name: hirakawa_sample, batch_size: 4, epochs: 100, save_every_steps: 1000, freeze_ZH_bert: False, freeze_JP_bert: False, freeze_EN_bert: False, freeze_style: False, freeze_decoder: False, use_jp_extra: False
04-03 07:20:00 |WARNING | train.py:103 | Step 1: Data/hirakawa_sample/models already exists, so copy it to backup to Data/hirakawa_sample/models_backup
04-03 07:25:02 |SUCCESS | train.py:132 | Step 1: initialization finished.
04-03 07:25:02 |  INFO  | train.py:137 | Step 2: start resampling...
04-03 07:25:02 |  INFO  | subprocess.py:23 | Running: resample.py -i Data/hirakawa_sample/raw -o Data/hirakawa_sample/wavs --num_processes 1 --sr 44100
04-03 07:25:06 | ERROR  | subprocess.py:33 | Error: resample.py -i Data/hira

In [ ]:
# セル7.7: config.json 設定（batch_size / fp16 / save_every_steps）
import json

config_path = 'Data/hirakawa_sample/config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

config['train']['batch_size'] = 4
config['train']['fp16_run'] = True
config['train']['save_every_steps'] = 200

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"batch_size: {config['train']['batch_size']}")
print(f"fp16_run: {config['train']['fp16_run']}")
print(f"save_every_steps: {config['train']['save_every_steps']}")
print('config.json 更新完了 ✅')

batch_size: 4
fp16_run: True
save_every_steps: 200
config.json 更新完了 ✅


In [ ]:
# セル8: 前処理（bert_gen / style_gen）
import subprocess

# bert_gen
bert_count = int(subprocess.check_output(
    'find Data/hirakawa_sample/ -name "*.bert.pt" | wc -l', shell=True
).decode().strip())
print(f'既存の .bert.pt: {bert_count} ファイル')

if bert_count < 180:
    print('bert_gen.py を実行します...')
    !python bert_gen.py --config Data/hirakawa_sample/config.json
else:
    print('✅ bert_gen 前処理済み → スキップ')

# style_gen（torchvision 競合回避のためアンインストール）
npy_count = int(subprocess.check_output(
    'find Data/hirakawa_sample/ -name "*.npy" | wc -l', shell=True
).decode().strip())
print(f'既存の .npy: {npy_count} ファイル')

if npy_count < 180:
    print('style_gen.py を実行します...')
    !pip uninstall torchvision -y -q
    !python style_gen.py --config Data/hirakawa_sample/config.json
else:
    print('✅ style_gen 前処理済み → スキップ')

# 最終確認
bert_count = int(subprocess.check_output('find Data/hirakawa_sample/ -name "*.bert.pt" | wc -l', shell=True).decode().strip())
npy_count = int(subprocess.check_output('find Data/hirakawa_sample/ -name "*.npy" | wc -l', shell=True).decode().strip())
print(f'.bert.pt: {bert_count} ファイル')
print(f'.npy:     {npy_count} ファイル')

既存の .bert.pt: 188 ファイル
✅ bert_gen 前処理済み → スキップ
既存の .npy: 188 ファイル
✅ style_gen 前処理済み → スキップ
.bert.pt: 188 ファイル
.npy:     188 ファイル


In [ ]:
# セル9: 学習実行
import os
os.environ['MKL_THREADING_LAYER'] = 'GNU'

!python train_ms_jp_extra.py \
    -c Data/hirakawa_sample/config.json \
    -m Data/hirakawa_sample

In [ ]:
# セル10: モデルを Drive にバックアップ
import os

BACKUP_PATH = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample_models'
os.makedirs(BACKUP_PATH, exist_ok=True)

!cp Data/hirakawa_sample/models/*.safetensors {BACKUP_PATH}/ 2>/dev/null || true
!cp Data/hirakawa_sample/config.json {BACKUP_PATH}/ 2>/dev/null || true

backed_up = os.listdir(BACKUP_PATH)
print(f'バックアップ完了: {backed_up} ✅')

In [ ]:
# セル11: 推論テスト
import subprocess

# 最新モデルを確認
models = !find Data/hirakawa_sample/models -name 'G_*.safetensors' | sort -V | tail -1
latest_model = models[0] if models else None
print(f'最新モデル: {latest_model}')

if latest_model:
    !python infer.py \
        --model_path {latest_model} \
        --config_path Data/hirakawa_sample/config.json \
        --text 'こんにちは、テストです。' \
        --output output_test.wav
    print('推論完了 ✅ → output_test.wav')
else:
    print('モデルが見つかりません。学習を完了してください。')